# Retail Sales Analytics -- Exploratory Data Analysis

**Dataset:** Supermarket Sales (Kaggle) -- 1000 transactions across 3 branches in Myanmar (Yangon, Mandalay, Naypyitaw).

This notebook walks through the exploratory analysis behind the `Retail-Sales-Analytics` project. The reusable, production-style logic lives in `src/`; this notebook exists to explore the data interactively, sanity-check assumptions, and preview the same charts the CLI/Streamlit app generate.


In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.data_cleaning import DataCleaner
from src.analysis import SalesAnalysis
from src.visualization import Visualizer

sns.set_theme(style="whitegrid", palette="Set2")
%matplotlib inline


## 1. Load Raw Data

In [2]:
loader = DataLoader("../data/supermarket_sales.csv")
raw_df = loader.load_data()
raw_df.head()


DATA LOADING SUMMARY
File          : supermarket_sales.csv
Shape         : 1000 rows x 17 columns
Missing cells : 0
Duplicate rows: 0
------------------------------------------------------------
Column dtypes:
Invoice ID                     str
Branch                         str
City                           str
Customer type                  str
Gender                         str
Product line                   str
Unit price                 float64
Quantity                     int64
Tax 5%                     float64
Total                      float64
Date                           str
Time                           str
Payment                        str
cogs                       float64
gross margin percentage    float64
gross income               float64
Rating                     float64
dtype: object
------------------------------------------------------------
Sample rows:
    Invoice ID Branch       City Customer type  Gender  \
0  750-67-8428      A     Yangon        Member  F

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


## 2. Data Cleaning & Feature Engineering

The raw dataset from Kaggle is already fairly clean (no nulls or duplicates), but the `DataCleaner`
class still runs a full defensive pipeline -- duplicate removal, missing value imputation, dtype
correction, invalid-row filtering, and column renaming -- so the pipeline is robust to messier
real-world exports. It also engineers new columns: `month`, `weekday`, `hour`, `sales_category`,
and `revenue_class`.

In [3]:
cleaner = DataCleaner(raw_df)
df = cleaner.clean()
df.head()


DATA CLEANING SUMMARY
Rows before cleaning : 1000
Rows after cleaning  : 1000
Rows removed         : 0
Columns (final)      : ['invoice_id', 'branch', 'city', 'customer_type', 'gender', 'product_line', 'unit_price', 'quantity', 'tax', 'total', 'date', 'time', 'payment', 'cogs', 'gross_margin_pct', 'gross_income', 'rating', 'month', 'weekday', 'day', 'hour', 'sales_category', 'revenue_class']



,invoice_id,branch,city,customer_type,gender,product_line,unit_price,quantity,tax,total,...,cogs,gross_margin_pct,gross_income,rating,month,weekday,day,hour,sales_category,revenue_class
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,...,522.83,4.761905,26.1415,9.1,January,Saturday,5,13,Medium,Premium
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,...,76.40,4.761905,3.8200,9.6,March,Friday,8,10,Medium,Low
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,...,324.31,4.761905,16.2155,7.4,March,Sunday,3,13,Medium,High
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,...,465.76,4.761905,23.2880,8.4,January,Sunday,27,20,Bulk,Premium
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,...,604.17,4.761905,30.2085,5.3,February,Friday,8,10,Medium,Premium


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   invoice_id        1000 non-null   str           
 1   branch            1000 non-null   category      
 2   city              1000 non-null   category      
 3   customer_type     1000 non-null   category      
 4   gender            1000 non-null   category      
 5   product_line      1000 non-null   category      
 6   unit_price        1000 non-null   float64       
 7   quantity          1000 non-null   int64         
 8   tax               1000 non-null   float64       
 9   total             1000 non-null   float64       
 10  date              1000 non-null   datetime64[us]
 11  time              1000 non-null   object        
 12  payment           1000 non-null   category      
 13  cogs              1000 non-null   float64       
 14  gross_margin_pct  1000 non-null   fl

In [5]:
df.describe()


,unit_price,quantity,tax,total,date,cogs,gross_margin_pct,gross_income,rating,day,hour
count,1000.000000,1000.000000,1000.000000,1000.000000,1000,1000.00000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000
mean,55.672130,5.510000,15.379369,322.966749,2019-02-14 00:05:45.600000,307.58738,4.761905,15.379369,6.97270,15.256000,14.910000
min,10.080000,1.000000,0.508500,10.678500,2019-01-01 00:00:00,10.17000,4.761905,0.508500,4.00000,1.000000,10.000000
25%,32.875000,3.000000,5.924875,124.422375,2019-01-24 00:00:00,118.49750,4.761905,5.924875,5.50000,8.000000,12.000000
50%,55.230000,5.000000,12.088000,253.848000,2019-02-13 00:00:00,241.76000,4.761905,12.088000,7.00000,15.000000,15.000000
75%,77.935000,8.000000,22.445250,471.350250,2019-03-08 00:00:00,448.90500,4.761905,22.445250,8.50000,23.000000,18.000000
max,99.960000,10.000000,49.650000,1042.650000,2019-03-30 00:00:00,993.00000,4.761905,49.650000,10.00000,31.000000,20.000000
std,26.494628,2.923431,11.708825,245.885335,NaN,234.17651,0.000000,11.708825,1.71858,8.693563,3.186857


## 3. Key Performance Indicators

In [6]:
analysis = SalesAnalysis(df)
kpis = analysis.calculate_kpis()
for k, v in kpis.items():
    print(f"{k:25s}: {v:,.2f}")


total_revenue            : 322,966.75
total_transactions       : 1,000.00
total_quantity_sold      : 5,510.00
average_order_value      : 322.97
median_order_value       : 253.85
std_order_value          : 245.76
average_rating           : 6.97
total_gross_income       : 15,379.37


## 4. Branch Analysis

In [7]:
branch_summary = analysis.branch_analysis()
branch_summary


,total_revenue,total_gross_income,avg_rating,transactions,avg_basket_size
branch,,,,,
C,110568.71,5265.18,7.07,328,5.58
A,106200.37,5057.16,7.03,340,5.47
B,106197.67,5057.03,6.82,332,5.48


In [8]:
viz = Visualizer(df, output_dir="../outputs/charts")
viz.plot_branch_sales(branch_summary)
plt.show()


## 5. Customer Analysis

In [9]:
customer_stats = analysis.customer_analysis()
customer_stats


{'gender_split': {'Female': 501, 'Male': 499},
 'membership_split': {'Member': 501, 'Normal': 499},
 'avg_spend_by_gender': {'Female': 335.1, 'Male': 310.79},
 'avg_rating_by_membership': {'Member': 6.94, 'Normal': 7.01},
 'avg_spend_by_membership': {'Member': 327.79, 'Normal': 318.12},
 'segment_counts': {'Member-Female': 261,
  'Member-Male': 240,
  'Normal-Female': 240,
  'Normal-Male': 259}}

In [10]:
viz.plot_customer_analysis()
plt.show()


## 6. Product Analysis

In [11]:
product_summary = analysis.product_analysis()
product_summary


,total_revenue,total_quantity,avg_rating,avg_unit_price,transactions
product_line,,,,,
Food and beverages,56144.84,952,7.11,56.01,174
Sports and travel,55122.83,920,6.92,56.99,166
Electronic accessories,54337.53,971,6.92,53.55,170
Fashion accessories,54305.90,902,7.03,57.15,178
Home and lifestyle,53861.91,911,6.84,55.32,160
Health and beauty,49193.74,854,7.00,54.85,152


In [12]:
pivot = analysis.product_branch_pivot()
viz.plot_product_analysis(product_summary, pivot)
plt.show()


## 7. Payment Method Analysis

In [13]:
payment_stats = analysis.payment_analysis()
payment_stats


{'counts': {'Ewallet': 345, 'Cash': 344, 'Credit card': 311},
 'revenue_by_payment': {'Ewallet': 109993.11,
  'Cash': 112206.57,
  'Credit card': 100767.07},
 'most_preferred': 'Ewallet'}

In [14]:
viz.plot_payment_distribution(payment_stats["counts"])
plt.show()


## 8. City Analysis

In [15]:
city_summary = analysis.city_analysis()
city_summary


,total_revenue,total_quantity,avg_rating,transactions
city,,,,
Naypyitaw,110568.71,1831,7.07,328
Yangon,106200.37,1859,7.03,340
Mandalay,106197.67,1820,6.82,332


In [16]:
viz.plot_city_sales(city_summary)
plt.show()


## 9. Time-Based Trends

In [17]:
time_stats = analysis.time_analysis()
print("Peak hour:", time_stats["peak_hour"])
print("Best weekday:", time_stats["best_weekday"])
time_stats["sales_by_hour"]


Peak hour: 19
Best weekday: Saturday


hour
10    31421.48
11    30377.33
12    26065.88
13    34723.23
14    30828.40
15    31179.51
16    25226.32
17    24445.22
18    26030.34
19    39699.51
20    22969.53
Name: total, dtype: float64

In [18]:
viz.plot_time_analysis(time_stats["sales_by_hour"], time_stats["sales_by_month"])
plt.show()


## 10. Ratings & Correlation Analysis

In [19]:
rating_stats = analysis.rating_analysis()
rating_stats["correlation_matrix"]


,quantity,total,gross_income,rating
quantity,1.000,0.706,0.706,-0.016
total,0.706,1.000,1.000,-0.036
gross_income,0.706,1.000,1.000,-0.036
rating,-0.016,-0.036,-0.036,1.000


In [20]:
viz.plot_rating_histogram()
plt.show()
viz.plot_correlation_matrix(rating_stats["correlation_matrix"])
plt.show()


## 11. NumPy Showcase: Normalized Branch Revenue

A quick demonstration of vectorised NumPy broadcasting -- min-max normalising branch revenue
onto a 0-1 scale without writing an explicit loop.

In [21]:
analysis.normalized_revenue_by_branch()


{'A': np.float64(0.001), 'B': np.float64(0.0), 'C': np.float64(1.0)}

## 12. Takeaways

- Revenue, product performance, and satisfaction don't always move together -- see the
  correlation matrix above.
- The full, reusable pipeline (cleaning -> analysis -> charts -> report) is implemented in
  `src/sales_analyzer.py` and can be run end-to-end with `python main.py --full` from the
  project root, or explored interactively via the Streamlit app in `frontend/app.py`.
